# Metoda Monte Carlo w symulacji portfela inwestycyjnego

W tym opracowaniu zajmuję się jednym z podstawowych problemów zarządzania ryzykiem: jak oszacować, ile może stracić portfel złożony z wielu aktywów, skoro ich przyszłe ceny są niepewne i wzajemnie powiązane. Pokażę, że to pytanie sprowadza się do całki wielokrotnej z funkcji wielu zmiennych, a metoda Monte Carlo pozwala ją przybliżyć niezależnie od liczby aktywów.

Struktura opracowania:

1. **Problem informatyczny** – opisuję, po co i dlaczego symuluję portfel.
2. **Matematyka** – przedstawiam funkcje wielu zmiennych, całki wielokrotne i wielowymiarowy rozkład normalny, a następnie uzasadniam, dlaczego Monte Carlo działa i jak duży jest jego błąd.
3. **Zastosowanie** – stosuję metodę do rzeczywistych notowań GPW i sprawdzam wyniki na danych, których model nie widział.

Kod funkcji pomocniczych znajduje się w pakiecie `portfolio_simulation` (katalog `src/`). W notebooku zostawiam tylko kod potrzebny do przeprowadzenia analizy.

## Konfiguracja

Importy i wszystkie ustawienia globalne (ziarna generatora, ścieżki, parametry portfela i testu) zbieram w jednej komórce, żeby wyniki były powtarzalne, a zmianę założeń można było zrobić w jednym miejscu.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats as st

from portfolio_simulation.backtest import rolling_var_backtest
from portfolio_simulation.data import download_yahoo, load_prices, log_returns
from portfolio_simulation.monte_carlo import mc_mean, value_at_risk
from portfolio_simulation.simulation import (
    calibrate,
    portfolio_value,
    sample_log_returns,
    simulate_asset_paths,
)

# --- Ustawienia globalne ---
SEED_DEMO = 42                           # przykład syntetyczny (rozdział 2)
SEED_REAL = 7                            # dane rzeczywiste (rozdział 3)

DATA_DIR = Path("..") / "data" / "raw"
TICKERS = ["pko", "pkn", "pzu", "kgh"]   # PKO BP, Orlen, PZU, KGHM (na Yahoo: PKO.WA itd.)

CAPITAL = 100_000                        # kapitał początkowy [PLN]
HORIZON = 21                             # horyzont prognozy [sesje], ok. miesiąc
N_SIMS = 50_000                          # liczba scenariuszy w rozdziale 3
ALPHA = 0.05                             # poziom ogona (VaR 95%)
BT_WINDOW = 500                          # okno kalibracji w backteście [sesje]

plt.rcParams["figure.figsize"] = (9, 4)


## 1. Problem informatyczny

### 1.1 Skąd się bierze problem

Każdy, kto inwestuje, czy to osoba odkładająca na emeryturę, czy fundusz zarządzający miliardami, musi odpowiedzieć nie tylko na pytanie „ile mogę zarobić?", ale także „ile mogę stracić?". Dla banków, funduszy i ubezpieczycieli to codzienna konieczność: od odpowiedzi zależy, ile kapitału trzeba trzymać w rezerwie i jak duże pozycje wolno zajmować. Standardową miarą ryzyka jest **Value at Risk (VaR)**, czyli strata, która w zadanym horyzoncie nie zostanie przekroczona z prawdopodobieństwem, powiedzmy, $95\%$.

Rozważam inwestora z portfelem akcji kilku dużych spółek giełdowych o łącznej wartości $100\,000$ PLN, który chce wiedzieć, jak bardzo jego majątek może zmaleć w ciągu najbliższego miesiąca. Nie pyta o prognozę ceny, bo tej nie da się rzetelnie podać. Pyta o to, **jak wygląda rozkład możliwych wyników** i jak gruby jest jego lewy ogon. Taki właśnie przypadek rozwiązuję w rozdziale 3.

### 1.2 Dlaczego prosta odpowiedź nie wystarcza

Najprostsza odpowiedź opiera się na historii: sprawdzam, jak portfel zachowywał się dotąd, i zakładam, że przyszłość będzie podobna. Takie podejście ma trzy słabości.

Po pierwsze, **średnia nie mówi nic o rozrzucie**. Dwa portfele o tym samym średnim zwrocie mogą mieć diametralnie różne ryzyko, a inwestora interesuje właśnie to, co dzieje się w złych scenariuszach, nie w przeciętnym.

Po drugie, **aktywa nie są niezależne**. Akcje banku i ubezpieczyciela często spadają jednocześnie, więc o ryzyku całości decyduje nie tylko ryzyko każdej spółki z osobna, ale też to, jak bardzo poruszają się razem. Współzależności sprawiają, że zróżnicowany portfel jest bezpieczniejszy.

Po trzecie, **historia to jedna realizacja**. Rynek wylosował dokładnie jedną ścieżkę spośród nieskończenie wielu możliwych. Potrzebuję więc modelu, który potrafi wygenerować **wiele możliwych przyszłości**, a nie powtarzać jedną znaną przeszłość.

### 1.3 Sformułowanie problemu

Portfel składa się z $n$ aktywów: mam $x_i$ sztuk $i$-tego, którego cena w chwili $T$ wyniesie $S_i(T)$. Wartość portfela to
$$
V(s_1,\dots,s_n)=\sum_{i=1}^{n}x_i s_i ,
$$
czyli **funkcja $n$ zmiennych** $V:\mathbb{R}^n\to\mathbb{R}$. Ceny $S=(S_1,\dots,S_n)$ są jednak zmiennymi losowymi o łącznym rozkładzie z gęstością $f$. Interesują mnie dwie wielkości: średnia wartość portfela i prawdopodobieństwo, że spadnie poniżej zadanego poziomu $v$. Obie są **całkami $n$-krotnymi**:
$$
\mathbb{E}[V]=\int_{\mathbb{R}^n}V(s)\,f(s)\,ds ,
\qquad
\mathbb{P}(V<v)=\int_{\{V(s)<v\}}f(s)\,ds .
$$
Ogólnie szukam $\mathbb{E}[g(S)]$ dla pewnej funkcji wypłaty $g$. Zadanie informatyczne brzmi więc: **zaprojektować algorytm, który dla zadanego modelu rozkładu cen i funkcji $g$ poda przybliżenie $\mathbb{E}[g(S)]$ z kontrolowanym błędem i w rozsądnym czasie.**

### 1.4 Dlaczego to zadanie jest trudne

Pierwsza trudność to **wymiar**. Realistyczny portfel ma kilkadziesiąt pozycji, więc całka jest kilkudziesięciowymiarowa. Druga: **rozkład wartości portfela nie ma prostej postaci**. Ceny są dodatnie i zwykle modeluje się je rozkładem log-normalnym, a suma zmiennych log-normalnych nie ma rozkładu opisanego prostym wzorem. Trzecia: interesują mnie **zdarzenia rzadkie** (ogon rozkładu), a im rzadsze zdarzenie, tym trudniej je oszacować. Czwarta: w praktyce wypłata bywa **nieliniowa lub zależna od ścieżki** (opcje, rebalansowanie portfela, strategie), i wtedy wzory zamknięte w ogóle nie istnieją.

Naturalny pomysł to policzyć całkę numerycznie, dzieląc każdą oś na $k$ punktów i sumując wartości w węzłach siatki. Liczba węzłów rośnie jednak jak $k^n$.

In [ ]:
k = 10  # punktów siatki na jedną oś
n_assets = [1, 2, 3, 5, 10, 30]

pd.DataFrame(
    {
        "liczba aktywów n": n_assets,
        "punkty siatki k^n": [f"{k**n:.0e}" for n in n_assets],
    }
)

Już dla $n=10$ potrzebuję $10^{10}$ obliczeń, a dla $n=30$ liczba $10^{30}$ przekracza możliwości jakiegokolwiek komputera. To tzw. **przekleństwo wymiarowości**. Nie pomaga szybszy sprzęt, bo problem rośnie wykładniczo z wymiarem.

### 1.5 Idea rozwiązania

Metoda Monte Carlo odwraca podejście: zamiast przeglądać całą przestrzeń, **losuję** scenariusze cen zgodnie z modelem i obserwuję, co się z nimi dzieje. Metoda powstała w latach czterdziestych XX wieku przy obliczeniach fizycznych, do finansów weszła w latach siedemdziesiątych i dziś jest standardowym narzędziem wyceny instrumentów i oceny ryzyka. Algorytm jest prosty:

1. losuję wektor cen $S$ z zadanego rozkładu,
2. liczę $g(S)$ (np. wartość portfela),
3. powtarzam kroki 1–2 $N$ razy,
4. uśredniam wyniki (dla $\mathbb{E}[g]$) albo biorę odpowiedni kwantyl (dla VaR).

Koszt jednego scenariusza rośnie z wymiarem tylko wielomianowo, a scenariusze są niezależne, więc obliczenia łatwo zrównoleglić. Cena jest taka, że wynik jest **losowy i przybliżony**, więc muszę umieć oszacować jego błąd. W rozdziale 2 wykazuję, że błąd maleje jak $1/\sqrt{N}$ **niezależnie od wymiaru $n$**. Szkielet metody mieści się w kilku liniach:

In [ ]:
def monte_carlo(sample_scenario, payoff, n):
    """Szkielet metody: średnia z n losowych scenariuszy."""
    return np.mean([payoff(sample_scenario()) for _ in range(n)])

W rozdziale 3 używam wersji wektorowej (NumPy), bo jest wielokrotnie szybsza. Pozostają trzy pytania: z jakiego rozkładu losować skorelowane ceny, skąd wiem, że to działa, i jak duży jest błąd. Odpowiadam na nie w rozdziale 2. W rozdziale 3 stosuję wyniki do rzeczywistych notowań GPW. Na koniec otrzymuję symulator rynku, w którym można testować strategie, także te wybierane przez agenta uczenia ze wzmocnieniem (to kolejny etap projektu).

## 2. Matematyka

W rozdziale 1 zostawiłem trzy rzeczy bez uzasadnienia: skąd biorę skorelowane ceny, dlaczego średnia z losowań zbliża się do całki i jak duży jest błąd. Porządkuję je w ośmiu krokach: portfel jako funkcja wielu zmiennych (2.1), wartość oczekiwana jako całka wielokrotna (2.2), wielowymiarowy rozkład normalny i losowanie skorelowanych cen (2.3), model cen i wzór dokładny (2.4), estymator Monte Carlo i jego błąd (2.5), VaR jako kwantyl (2.6), wzór przybliżony i wrażliwość na parametry (2.7) oraz sprawdzenie liczbowe (2.8).

### 2.1 Portfel jako funkcja wielu zmiennych

Wartość portfela
$$
V(s) = x^\top s = \sum_{i=1}^n x_i s_i, \qquad s=(s_1,\dots,s_n)\in\mathbb{R}^n,
$$
ma pochodne cząstkowe $\dfrac{\partial V}{\partial s_i} = x_i$, a więc gradient $\nabla V = x$ jest stały. Liczba sztuk $x_i$ jest zatem wrażliwością portfela na cenę $i$-tego aktywa. Zbiory poziomu $\{V = v\}$ są hiperpłaszczyznami prostopadłymi do $x$, a zdarzenie „portfel wart mniej niż $v$" to półprzestrzeń $\{V<v\}$.

> **Uwaga.** Liniowość $V$ upraszcza rachunki (dzięki niej w 2.4 dostanę wzór dokładny), ale metoda Monte Carlo jej nie wymaga. Dla opcji kupna wypłata $\max(s-K,0)$ nie jest nawet różniczkowalna w punkcie $s=K$, a symulacja działa bez zmian.

### 2.2 Wartość oczekiwana jako całka wielokrotna

Jeśli wektor cen $S$ ma gęstość $f:\mathbb{R}^n\to[0,\infty)$, to dla funkcji $g:\mathbb{R}^n\to\mathbb{R}$
$$
\mathbb{E}[g(S)] = \int_{\mathbb{R}^n} g(s)\,f(s)\,ds .
$$
Z twierdzenia Fubiniego tę całkę $n$-krotną można liczyć jako iterowaną, czyli jako $n$ całek jednowymiarowych jedna w drugiej. Dokładnie tak działa siatka z 1.4 i stąd bierze się $k^n$ węzłów. Wszystkie moje pytania mają tę samą postać:

- średnia wartość portfela: $g = V$,
- prawdopodobieństwo straty: $g(s) = \mathbf{1}_{\{V(s)<v\}}$, czyli funkcja charakterystyczna półprzestrzeni, równa $1$ gdy $V(s)<v$ i $0$ w przeciwnym razie.

Wystarczy więc umieć przybliżać $\mathbb{E}[g(S)]$ dla dowolnej $g$. Idea Monte Carlo polega na tym, że jeśli potrafię losować z gęstości $f$, to całkę zastępuję średnią wartości $g$ w wylosowanych punktach (2.5).

### 2.3 Wielowymiarowy rozkład normalny

> **Definicja.** Niech $m\in\mathbb{R}^n$, a $\Sigma$ będzie macierzą symetryczną i dodatnio określoną. Wektor $Y$ ma rozkład $N(m,\Sigma)$, jeśli ma gęstość
> $$
f(y) = \frac{1}{(2\pi)^{n/2}\sqrt{\det\Sigma}}\exp\!\Big(-\tfrac12 (y-m)^\top \Sigma^{-1}(y-m)\Big).
$$

Wykładnik to $-\tfrac12 q(y)$, gdzie $q(y)=(y-m)^\top\Sigma^{-1}(y-m)$ jest **formą kwadratową** zmiennych $y_1,\dots,y_n$. Ponieważ $\Sigma^{-1}$ jest dodatnio określona, hesjan formy $q$ (równy $2\Sigma^{-1}$) jest dodatnio określony, więc $q$ ma jedyne minimum w punkcie $m$, a gęstość $f$ jedyne maksimum. Poziomice gęstości $\{q=\text{const}\}$ są elipsoidami i taki kształt mają „chmury" skorelowanych cen.

W praktyce komputer generuje niezależne liczby o rozkładzie $N(0,1)$ (generator pseudolosowy), a zależności między aktywami wprowadza dopiero przekształcenie liniowe. Umożliwia to rozkład Cholesky'ego:

> **Twierdzenie (Cholesky).** Dla macierzy $\Sigma$ jak wyżej istnieje dolnotrójkątna macierz $L$ o dodatnich elementach na przekątnej, taka że $\Sigma = LL^\top$. Jeśli $Z\sim N(0,I_n)$ (niezależne, standardowe normalne), to
> $$
Y = m + LZ \sim N(m,\Sigma).
$$

**Dowód** (zamiana zmiennych w całce wielokrotnej). Gęstość $Z$ to $\varphi(z) = (2\pi)^{-n/2}e^{-\frac12 z^\top z}$. Odwzorowanie $y = m+Lz$ jest afiniczne i odwracalne, a $z = L^{-1}(y-m)$. Jakobian odwzorowania odwrotnego wynosi $\det L^{-1}$, stąd
$$
f_Y(y) = \varphi\big(L^{-1}(y-m)\big)\,\frac{1}{|\det L|}.
$$
Ponieważ $\det\Sigma = (\det L)^2$, mamy $|\det L| = \sqrt{\det\Sigma}$. Ponadto $\Sigma^{-1} = (LL^\top)^{-1} = L^{-\top}L^{-1}$, więc
$$
z^\top z = (y-m)^\top L^{-\top}L^{-1}(y-m) = (y-m)^\top\Sigma^{-1}(y-m).
$$
Podstawiając, otrzymuję gęstość z definicji. $\blacksquare$

**Przykład ($n=2$).** Dla odchyleń $\sigma_1,\sigma_2$ i korelacji $\rho$
$$
L = \begin{pmatrix}\sigma_1 & 0\\ \rho\sigma_2 & \sigma_2\sqrt{1-\rho^2}\end{pmatrix},
\qquad
\begin{aligned}
Y_1 &= m_1 + \sigma_1 Z_1,\\
Y_2 &= m_2 + \sigma_2\big(\rho Z_1 + \sqrt{1-\rho^2}\,Z_2\big).
\end{aligned}
$$
Druga współrzędna to „$\rho$ części pierwszego szumu plus dopełnienie niezależnym szumem". Bezpośrednio sprawdzam, że $LL^\top$ daje $\sigma_1^2$, $\rho\sigma_1\sigma_2$ i $\sigma_2^2$.

Do ilustracji w całym rozdziale 2 używam syntetycznego portfela trzech aktywów. Poniżej definiuję jego parametry.

In [ ]:
rng = np.random.default_rng(SEED_DEMO)

s0 = np.array([100.0, 50.0, 200.0])      # ceny początkowe
shares = np.array([10.0, 20.0, 5.0])     # liczba sztuk
mu = np.array([0.0004, 0.0003, 0.0005])  # średnie dzienne zwroty
vol = np.array([0.012, 0.008, 0.015])    # dzienne odchylenia standardowe
corr = np.array([
    [1.0, 0.3, 0.5],
    [0.3, 1.0, 0.2],
    [0.5, 0.2, 1.0],
])
cov = np.outer(vol, vol) * corr          # macierz kowariancji

n_days, n_paths = 252, 10_000            # rok handlowy, liczba scenariuszy

Sprawdzam liczbowo, że macierz $LL^\top$ odtwarza $\Sigma$, a wektory $Y=m+LZ$ mają zadaną kowariancję.

In [ ]:
L = np.linalg.cholesky(cov)
print("L @ L.T == cov:", np.allclose(L @ L.T, cov))

r = sample_log_returns(mu, cov, (200_000,), rng)
print("Empiryczna kowariancja / docelowa (powinno być ok. 1):")
print(np.round(np.cov(r.T) / cov, 3))

Stosunek empirycznej kowariancji do docelowej mieści się w granicach $1{,}00$–$1{,}01$. Odchylenia rzędu procenta odpowiadają błędowi losowemu próby ($200\,000$ wektorów). Potwierdza to, że $Y=m+LZ$ odtwarza zadaną macierz $\Sigma$.

### 2.4 Model cen i wzór dokładny

Przyjmuję, że dzienne log-zwroty $r_t=\ln (S_t/S_{t-1})\in\mathbb{R}^n$ są niezależne i mają rozkład $N(d,\Sigma)$ z dryfem $d = \mu - \tfrac12\operatorname{diag}(\Sigma)$, gdzie $\sigma_i^2=\Sigma_{ii}$ jest dzienną wariancją $i$-tego aktywa. Ceny są wtedy dodatnie, a
$$
S_T = s_0\odot\exp\Big(\sum_{t=1}^T r_t\Big),
$$
gdzie $\odot$ i $\exp$ działają po współrzędnych. Suma $T$ niezależnych wektorów normalnych jest normalna, czyli $\ln S_T\sim N(\ln s_0 + Td,\; T\Sigma)$, a więc $S_T$ ma rozkład log-normalny.

> **Pytanie.** Skoro znam rozkład $S_T$, po co symulować całe ścieżki? Dla $\mathbb{E}[V_T]$ rzeczywiście nie trzeba. Ścieżki są jednak niezbędne, gdy wypłata zależy od drogi cen, a w części dotyczącej agenta RL decyzje będą podejmowane dzień po dniu.

Dla $Y\sim N(m,s^2)$ zachodzi $\mathbb{E}[e^Y]=e^{m+s^2/2}$. Dowodzę tego przez dopełnienie do kwadratu w wykładniku, $y-\frac{(y-m)^2}{2s^2}=m+\frac{s^2}{2}-\frac{(y-m-s^2)^2}{2s^2}$, skąd
$$
\mathbb{E}[e^Y]=\int_{\mathbb{R}} e^{y}\,\frac{e^{-(y-m)^2/(2s^2)}}{s\sqrt{2\pi}}\,dy
= e^{m+s^2/2}\int_{\mathbb{R}}\frac{e^{-(y-m-s^2)^2/(2s^2)}}{s\sqrt{2\pi}}\,dy = e^{m+s^2/2},
$$
bo ostatnia całka to całka gęstości $N(m+s^2,s^2)$, równa $1$. Stosuję to do $Y=\ln S_i(T)$, która ma średnią $\ln s_{0,i}+T d_i$ i wariancję $T\sigma_i^2$:
$$
\mathbb{E}[S_i(T)] = s_{0,i}\,e^{T(\mu_i-\sigma_i^2/2) + T\sigma_i^2/2} = s_{0,i}\,e^{\mu_i T},
\qquad
\mathbb{E}[V_T] = \sum_{i=1}^n x_i s_{0,i}\,e^{\mu_i T}.
$$
Poprawka $-\tfrac12\sigma_i^2$ w dryfie jest dobrana właśnie po to, by średnia cena rosła o czynnik $e^{\mu_i T}$. Mam więc **wzór dokładny**, z którym porównam symulację w 2.8. Ten sam rachunek dla $Y=\ln S_i+\ln S_j$ daje także $\operatorname{Cov}(S_i(T),S_j(T))=\mathbb{E}[S_i(T)]\,\mathbb{E}[S_j(T)]\,(e^{T\Sigma_{ij}}-1)$, a więc dokładną wariancję $V_T$. Będzie mi potrzebna do oceny błędu.

### 2.5 Estymator Monte Carlo i jego błąd

Niech $X=g(S)$, a $X_1,\dots,X_N$ będą niezależnymi kopiami tej zmiennej (wartościami $g$ w $N$ wylosowanych scenariuszach). Estymator Monte Carlo wielkości $J=\mathbb{E}[X]=\int g f\,ds$ to średnia
$$
\hat J_N = \frac1N\sum_{k=1}^N X_k ,
$$
czyli dokładnie algorytm z 1.5. Dwa twierdzenia z rachunku prawdopodobieństwa wyjaśniają, dlaczego działa i jak dokładnie.

> **Twierdzenie (mocne prawo wielkich liczb).** Jeśli $\mathbb{E}|X|<\infty$, to $\hat J_N\to J$ z prawdopodobieństwem 1. Ponadto $\mathbb{E}[\hat J_N]=J$ dla każdego $N$ (estymator jest nieobciążony).

> **Twierdzenie (centralne twierdzenie graniczne).** Jeśli $\sigma^2=\operatorname{Var}X<\infty$, to
> $$
> \sqrt N\,\frac{\hat J_N-J}{\sigma}\;\longrightarrow\; N(0,1) \quad\text{według rozkładu}.
> $$

Z niezależności $X_k$ wynika, że $\operatorname{Var}(\hat J_N)=\sigma^2/N$, czyli odchylenie standardowe estymatora wynosi dokładnie $\sigma/\sqrt N$, a CLT dodaje, że błąd ma w przybliżeniu rozkład normalny. Nieznane $\sigma$ zastępuję estymatorem $\hat\sigma^2=\frac1{N-1}\sum_k (X_k-\hat J_N)^2$ i dostaję przybliżony 95% przedział ufności
$$
\hat J_N \pm 1{,}96\,\hat\sigma/\sqrt N .
$$
Funkcja `mc_mean` zwraca średnią i błąd standardowy $\hat\sigma/\sqrt N$, a przedział buduję z $1{,}96$ takich błędów. Wniosek praktyczny: stukrotne zwiększenie $N$ zmniejsza błąd dziesięciokrotnie.

**Rola wymiaru.** Wymiar $n$ nie wchodzi do tempa $1/\sqrt N$. Wpływa tylko na koszt wylosowania jednego $X_k$ (generowanie wektora i mnożenie przez $L$) oraz na stałą $\sigma$, która zależy od $g$. Dla porównania oceniam błąd kwadratury na siatce o $N=k^n$ węzłach (reguła punktu środkowego, $f\in C^2$). Rozwijam $f$ w komórce o boku $h$ wokół jej środka $c$ (wzór Taylora dla funkcji wielu zmiennych):
$$
f(s)=f(c)+\nabla f(c)^\top(s-c)+\tfrac12 (s-c)^\top H_f(c)\,(s-c)+\dots
$$
Człon liniowy całkuje się do zera po symetrycznej komórce, więc błąd na jednej komórce to $O(h^{n+2})$. Komórek jest $h^{-n}$, więc błąd łączny to $O(h^2)=O(N^{-2/n})$, bo $h=N^{-1/n}$. Dla $n=1$ to $N^{-2}$, dla $n=4$ tyle samo co Monte Carlo ($N^{-1/2}$), a dla $n=30$ już tylko $N^{-1/15}$. Powyżej czterech wymiarów Monte Carlo wygrywa. Metody wyższego rzędu przesuwają ten próg, ale przy kilkudziesięciu wymiarach wniosek się nie zmienia.

### 2.6 Value at Risk jako kwantyl

Dla $g=\mathbf 1_{\{V<v\}}$ estymator $\hat p$ to po prostu odsetek scenariuszy, w których $V<v$. Ponieważ $\operatorname{Var}\mathbf 1=p(1-p)$, błąd standardowy wynosi $\sqrt{p(1-p)/N}$, a błąd względny $\approx\sqrt{(1-p)/(Np)}$. Dla $p=5\%$ i $N=10^4$ to około $4{,}4\%$, a dla $p=0{,}1\%$ już około $31\%$. Im dalej w ogonie rozkładu, tym więcej scenariuszy potrzeba.

$\mathrm{VaR}_\alpha = V_0-q_\alpha$, gdzie $q_\alpha$ jest $\alpha$-kwantylem rozkładu $V_T$. W symulacji zastępuję go kwantylem empirycznym, czyli odpowiednią statystyką pozycyjną z $N$ wyników. Kwantyl nie jest średnią, więc CLT dla średniej nie opisuje jego błędu wprost, ale wynik ma podobną postać: jeśli gęstość $f_V$ rozkładu $V_T$ jest dodatnia w $q_\alpha$, to
$$
\operatorname{SE}(\hat q_\alpha)\approx\frac{\sqrt{\alpha(1-\alpha)/N}}{f_V(q_\alpha)} .
$$
Błąd również maleje jak $1/\sqrt N$, ale jest tym większy, im rzadszy jest ogon i im mniejsza gęstość w punkcie $q_\alpha$. Gęstości $f_V$ nie znam, dlatego w praktyce błąd VaR oszacuję bootstrapem (3.4).

### 2.7 Wzór przybliżony i wrażliwość na parametry

**Wzór przybliżony na VaR.** Niech $w$ będzie wektorem udziałów wartościowych portfela. Dzienny zwrot portfela ma wariancję $\sigma_d^2=w^\top\Sigma w$ (znowu forma kwadratowa) i średnią $\mu_d=w^\top\mu$. Dla horyzontu $H$ sesji przybliżam zwrot portfela rozkładem normalnym $N(H\mu_d,\,H\sigma_d^2)$, skąd
$$
\mathrm{VaR}_\alpha\approx V_0\big(z_{1-\alpha}\,\sigma_d\sqrt H-H\mu_d\big),
$$
gdzie $z_{1-\alpha}$ to kwantyl standardowego rozkładu normalnego rzędu $1-\alpha$ ($1{,}645$ dla $\alpha=5\%$). Wzór jest szybki, ale zakłada normalność zwrotu portfela, więc posłuży jako punkt odniesienia dla Monte Carlo w 3.4.

**Wrażliwość na parametry.** Wartość oczekiwana zależy od $\mu$ przez $\mathbb{E}[V_T]=\sum_i x_i s_{0,i}e^{\mu_i T}$, więc
$$
\frac{\partial\,\mathbb{E}[V_T]}{\partial\mu_i}=x_i s_{0,i}\,T\,e^{\mu_i T}.
$$
Z różniczki zupełnej błąd estymacji $\delta\mu$ przenosi się na wynik jako $\delta\mathbb{E}[V_T]\approx\sum_i \frac{\partial\mathbb{E}[V_T]}{\partial\mu_i}\,\delta\mu_i$. Wykorzystam to w 3.4: błąd $\hat\mu$ jest dużo większy niż błąd Monte Carlo.

### 2.8 Sprawdzenie liczbowe na przykładzie syntetycznym

Sprawdzam teraz, czy Monte Carlo odtwarza wzór dokładny z 2.4. Symuluję $10\,000$ pełnych ścieżek portfela z 2.3 na horyzoncie roku ($252$ sesje) i wyznaczam wartość początkową, średnią wartość po roku i VaR.

In [ ]:
paths = simulate_asset_paths(s0, mu, cov, n_days, n_paths, rng)
values = portfolio_value(paths, shares)  # (n_paths, n_days)
v0 = float(shares @ s0)
final = values[:, -1]

print(f"Wartość początkowa portfela: {v0:,.0f}")
print(f"Średnia wartość po roku:     {final.mean():,.0f}")
print(f"VaR 95% (strata):            {value_at_risk(final, v0, 0.05):,.0f}")

In [ ]:
def exact_mean_std(x, s0, mu, cov, T):
    """Dokładna wartość oczekiwana i odchylenie standardowe V_T = x^T S_T (rozkład log-normalny z 2.4)."""
    m = s0 * np.exp(mu * T)                         # E[S_i(T)]
    c = np.outer(m, m) * (np.exp(T * cov) - 1.0)    # Cov(S_i(T), S_j(T))
    return x @ m, np.sqrt(x @ c @ x)


mean_exact, sd_exact = exact_mean_std(shares, s0, mu, cov, n_days)
print(f"Wzór dokładny:  E[V_T] = {mean_exact:,.1f},  odchylenie standardowe = {sd_exact:,.0f}")
print(f"Błąd standardowy średniej z {n_paths:,} scenariuszy: {sd_exact / np.sqrt(n_paths):.1f}")


Wzór dokładny daje $3\,318{,}9$, a symulacja $3\,322$. Różnica ($\approx 3$) jest mniejsza od błędu standardowego ($4{,}8$), a tym bardziej od półszerokości przedziału ufności ($1{,}96\cdot 4{,}8\approx 9{,}4$), więc wynik jest zgodny z CLT. Poniżej przedstawiam przykładowe scenariusze i rozkład wartości portfela po roku.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(values[:50].T, alpha=0.4, lw=0.8)
ax1.axhline(v0, color="black", ls="--", lw=1)
ax1.set(title="50 przykładowych scenariuszy", xlabel="dzień", ylabel="wartość portfela")

ax2.hist(final, bins=60, alpha=0.8)
ax2.axvline(v0, color="black", ls="--", label="start")
ax2.axvline(np.quantile(final, 0.05), color="red", label="5% kwantyl")
ax2.set(title="Rozkład wartości portfela po roku", xlabel="wartość portfela")
ax2.legend()

plt.tight_layout()
plt.show()

Rozkład jest skośny w prawo, jak przystało na rozkład log-normalny: wartość portfela może wzrosnąć bardzo mocno, a spaść najwyżej do zera. Dlatego o VaR decyduje lewy ogon, a nie średnia. Szacuję jeszcze prawdopodobieństwo, że po roku portfel jest wart mniej niż na starcie.

In [ ]:
p_hat, p_se = mc_mean((final < v0).astype(float))
print(f"P(wartość portfela < start) = {p_hat:.3f} ± {1.96 * p_se:.3f}")

Mimo dodatniego dryfu (średnia wartość portfela rośnie o ok. $10{,}7\%$ w skali roku) w mniej więcej co czwartym scenariuszu portfel po roku jest wart **mniej niż na starcie**. Dlatego sama wartość oczekiwana nie wystarcza do oceny inwestycji.

Na koniec sprawdzam empirycznie tempo zbieżności $1/\sqrt N$ i jego niezależność od wymiaru. Dla portfeli o $n=3$ i $n=30$ aktywach (wspólna korelacja $0{,}3$) losuję $S_T$ bezpośrednio z rozkładu log-normalnego z 2.4. Błąd względny średniej (pierwiastek z błędu średniokwadratowego ze $100$ powtórzeń) porównuję z teoretycznym $\sigma/\sqrt N$, które znam dokładnie dzięki wzorom z 2.4.

In [ ]:
def lognormal_terminal(s0, mu, cov, T, size, rng):
    """Losuje wektory cen S_T bezpośrednio z rozkładu log-normalnego z 2.4 (bez symulacji ścieżek)."""
    L = np.linalg.cholesky(cov)
    drift = mu - 0.5 * np.diag(cov)
    z = rng.standard_normal((size, len(s0)))
    return s0 * np.exp(T * drift + np.sqrt(T) * z @ L.T)


def make_portfolio(n, rng):
    """Syntetyczny portfel n aktywów: wspólna korelacja 0,3 i zróżnicowane zmienności."""
    vol = rng.uniform(0.008, 0.018, n)
    corr = np.full((n, n), 0.3)
    np.fill_diagonal(corr, 1.0)
    return np.full(n, 100.0), np.ones(n), np.full(n, 0.0004), np.outer(vol, vol) * corr


ns = np.unique(np.logspace(2, 5, 13).astype(int))   # liczby scenariuszy N
n_reps = 100                                        # powtórzenia eksperymentu
rng_conv = np.random.default_rng(SEED_DEMO)

fig, ax = plt.subplots()
for n_dim, marker in [(3, "o"), (30, "s")]:
    s0_c, x_c, mu_c, cov_c = make_portfolio(n_dim, rng_conv)
    mean_c, sd_c = exact_mean_std(x_c, s0_c, mu_c, cov_c, n_days)

    errors = []
    for _ in range(n_reps):
        v = lognormal_terminal(s0_c, mu_c, cov_c, n_days, ns[-1], rng_conv) @ x_c
        running_mean = np.cumsum(v) / np.arange(1, v.size + 1)   # średnia z pierwszych N scenariuszy
        errors.append(running_mean[ns - 1] - mean_c)
    rmse = np.sqrt(np.mean(np.square(errors), axis=0)) / mean_c

    line, = ax.loglog(ns, sd_c / mean_c / np.sqrt(ns), "-", label=f"n = {n_dim}: teoria $\\sigma/\\sqrt{{N}}$")
    ax.loglog(ns, rmse, marker, color=line.get_color(), label=f"n = {n_dim}: symulacja")

ax.set(title="Błąd względny estymatora Monte Carlo średniej", xlabel="liczba scenariuszy N", ylabel="błąd względny (RMSE)")
ax.legend()
plt.tight_layout()
plt.show()


Punkty układają się wzdłuż prostych o nachyleniu $-\tfrac12$ zarówno dla $n=3$, jak i dla $n=30$, zgodnie z teorią. Wymiar zmienia tylko stałą $\sigma$ (dla $30$ aktywów jest mniejsza dzięki dywersyfikacji), a nie tempo zbieżności. W obrębie jednego powtórzenia punkty dla kolejnych $N$ korzystają z tych samych scenariuszy (średnie kroczące), więc nie są niezależne. Niezależne są dopiero powtórzenia. Tak wygląda sedno przewagi Monte Carlo nad siatką z 1.4: stukrotne zwiększenie $N$ zmniejsza błąd dziesięciokrotnie niezależnie od liczby aktywów.

## 3. Zastosowanie na rzeczywistych danych

Model z rozdziału 2 stosuję do portfela czterech spółek z GPW: PKO BP, Orlen, PZU i KGHM. To duże spółki z różnych sektorów (bank, paliwa, ubezpieczenia, miedź), więc korelacje między nimi są umiarkowane. Kapitał wynosi 100 000 PLN i dzielę go po równo. Wykonuję trzy kroki:

1. kalibruję parametry $\mu$ i $\Sigma$ na historycznych notowaniach,
2. sprawdzam, czy założenia modelu (normalność, niezależność zwrotów) są spełnione,
3. liczę VaR metodą Monte Carlo i **weryfikuję go na danych, których model nie widział** (backtest).

### 3.1 Dane

Źródłem są dzienne ceny zamknięcia z Yahoo Finance, które pobieram biblioteką `yfinance`. Biblioteka korzysta z nieoficjalnego interfejsu i bywa niestabilna (limity zapytań), dlatego pliki CSV zapisuję w `data/raw/` i trzymam w repozytorium. Dzięki temu notebook da się uruchomić bez dostępu do Yahoo. Ceny są skorygowane o splity i dywidendy, więc analizuję zwroty całkowite. Dane obejmują sesje od 1 stycznia 2019 r. do 29 października 2026 r. (1980 sesji).

In [ ]:
# Pobieram tylko brakujące pliki, żeby nie obciążać Yahoo przy każdym uruchomieniu.
for ticker in TICKERS:
    if not (DATA_DIR / f"{ticker}.csv").exists():
        download_yahoo(ticker, DATA_DIR)


In [ ]:
prices = load_prices(TICKERS, DATA_DIR)
print(prices.index.min().date(), "→", prices.index.max().date(), f"({len(prices)} sesji)")

(prices / prices.iloc[0]).plot(figsize=(10, 4), title="Ceny znormalizowane (start = 1)")
plt.show()

### 3.2 Log-zwroty i kalibracja

Z cen liczę dzienne log-zwroty $r_t=\ln(S_t/S_{t-1})$. Parametry modelu estymuję klasycznymi wzorami (dla niezależnych zwrotów normalnych są to estymatory największej wiarygodności, z dokładnością do dzielnika w kowariancji):
$$
\hat d=\frac1T\sum_{t=1}^T r_t,\qquad
\hat\Sigma=\frac1{T-1}\sum_{t=1}^T (r_t-\hat d)(r_t-\hat d)^\top,\qquad
\hat\mu=\hat d+\tfrac12\operatorname{diag}(\hat\Sigma).
$$
Ostatni wzór odwraca poprawkę Itô z 2.4. Funkcja `calibrate` wykonuje dokładnie te trzy kroki.

In [ ]:
returns = log_returns(prices)
mu_hat, cov_hat = calibrate(returns.to_numpy())

summary_stats = pd.DataFrame(
    {
        "średni zwrot roczny": returns.mean() * 252,
        "zmienność roczna": returns.std() * np.sqrt(252),
    }
)
print(summary_stats.round(3))
print()
print("Macierz korelacji:")
print(returns.corr().round(2))

„Średni zwrot roczny" w tabeli to średni dzienny log-zwrot pomnożony przez $252$, czyli $252\,\hat d$, a nie $\hat\mu$.

In [ ]:
weights = np.full(len(TICKERS), 1 / len(TICKERS))

sigma_p = np.sqrt(weights @ cov_hat @ weights * 252)
print(f"Zmienność portfela: {sigma_p:.1%}, średnia zmienność spółek: {returns.std().mean() * np.sqrt(252):.1%}")

Zmienność portfela jest wyraźnie niższa od średniej zmienności jego składników. To efekt **dywersyfikacji**: wariancja portfela to forma kwadratowa $w^\top\Sigma w$, w której wyrazy pozadiagonalne (kowariancje) są mniejsze niż iloczyn odchyleń, o ile korelacje są poniżej $1$. Najsilniej skorelowana para (PKO BP i PZU, ok. $0{,}67$) to dwie spółki sektora finansowego.

### 3.3 Czy założenia modelu są spełnione?

Model zakłada, że log-zwroty mają rozkład normalny. W danych rynkowych to założenie zwykle nie jest spełnione: empiryczne zwroty mają **grubsze ogony** niż rozkład normalny, a zmienność przychodzi w skupiskach. Sprawdzam to na dwa sposoby: przez nadmiarową kurtozę (dla rozkładu normalnego równą $0$) i testem Jarque'a–Bery,
$$
JB=\frac T6\Big(S^2+\tfrac14 K^2\Big)\;\sim\;\chi^2(2)\ \text{przy normalności},
$$
gdzie $S$ to skośność, a $K$ nadmiarowa kurtoza próby. Konsekwencja grubych ogonów jest taka, że model normalny może zaniżać prawdopodobieństwo dużych strat, a więc i VaR.

In [ ]:
jb = {col: st.jarque_bera(returns[col]) for col in returns}

normality = pd.DataFrame(
    {
        "kurtoza nadmiarowa": returns.kurt().round(2),
        "statystyka JB": [f"{jb[col].statistic:,.0f}" for col in returns],
        "p-value": ["<1e-300" if jb[col].pvalue < 1e-300 else f"{jb[col].pvalue:.1e}" for col in returns],
    }
)
print(normality)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
st.probplot(returns["pko"], dist="norm", plot=ax1)
ax1.set_title("Wykres kwantyl–kwantyl: PKO BP vs normalny")

ax2.hist(returns["pko"], bins=80, density=True, alpha=0.7)
x = np.linspace(returns["pko"].min(), returns["pko"].max(), 400)
ax2.plot(x, st.norm.pdf(x, returns["pko"].mean(), returns["pko"].std()))
ax2.set_title("Dzienne log-zwroty PKO BP")

plt.tight_layout()
plt.show()


Normalność jest odrzucana dla wszystkich spółek (nadmiarowa kurtoza od ok. $2$ do ok. $6{,}5$ dla PKO BP). Przy prawie $2000$ obserwacji test Jarque'a–Bery wychwytuje nawet niewielkie odchylenia, więc ważniejszy jest wykres kwantyl–kwantyl: obserwacje z lewego ogona leżą wyraźnie poniżej prostej. Najgorsze dni są znacznie gorsze, niż przewiduje rozkład normalny.

### 3.4 Symulacja portfela

Portfel ma kapitał $100\,000$ PLN podzielony po równo na cztery spółki (ułamkowe liczby akcji, bez prowizji). Symuluję $50\,000$ scenariuszy na horyzoncie $21$ sesji (około miesiąca), startując z ostatnich notowań.

> **Uwaga.** Wartość oczekiwana zależy od $\hat\mu$, a ten estymator jest bardzo niepewny: błąd średniego rocznego zwrotu wynosi około $\sigma/\sqrt{\text{liczba lat}}$, czyli kilka–kilkanaście punktów procentowych. Na krótkim horyzoncie VaR zależy głównie od $\Sigma$, które estymuję znacznie dokładniej.

In [ ]:
s0_real = prices.iloc[-1].to_numpy()
shares_real = CAPITAL * weights / s0_real

rng = np.random.default_rng(SEED_REAL)
paths_real = simulate_asset_paths(s0_real, mu_hat, cov_hat, HORIZON, N_SIMS, rng)
final_real = portfolio_value(paths_real, shares_real)[:, -1]

mean, se = mc_mean(final_real)
var95 = value_at_risk(final_real, CAPITAL, ALPHA)
print(f"Oczekiwana wartość za {HORIZON} sesji: {mean:,.0f} PLN (± {1.96 * se:,.0f})")
print(f"VaR {1 - ALPHA:.0%} ({HORIZON} sesji): {var95:,.0f} PLN ({var95 / CAPITAL:.1%} kapitału)")

plt.hist(final_real, bins=80, alpha=0.8)
plt.axvline(CAPITAL, color="black", ls="--", label="start")
plt.axvline(CAPITAL - var95, color="red", label=f"{ALPHA:.0%} kwantyl")
plt.xlabel("wartość portfela [PLN]")
plt.legend()
plt.show()


Porównuję teraz dwa źródła błędu oraz wzór przybliżony z 2.7. Wszystkie wielkości podaję jako błędy standardowe (1 SE), żeby dało się je zestawić.

In [ ]:
years = len(returns) / 252
se_mu_period = sigma_p / np.sqrt(years) * HORIZON / 252    # błąd std. średniego zwrotu portfela w horyzoncie
sigma_h = sigma_p * np.sqrt(HORIZON / 252)                 # zmienność portfela w horyzoncie
var_delta = CAPITAL * (st.norm.ppf(1 - ALPHA) * sigma_h - (weights @ mu_hat) * HORIZON)

print(f"Błąd Monte Carlo (1 SE):        ±{se:,.0f} PLN")
print(f"Niepewność estymacji μ (1 SE):  ≈ {CAPITAL * se_mu_period:,.0f} PLN")
print(f"VaR wzorem przybliżonym:        {var_delta:,.0f} PLN")


**Dwa źródła błędu.** Błąd Monte Carlo (rzędu kilkudziesięciu PLN) mogę dowolnie zmniejszyć, zwiększając $N$. Niepewność estymacji $\hat\mu$ (rzędu kilkuset PLN) wynika z ograniczonej ilości danych i symulacja jej nie usuwa. Wzór z różniczką zupełną z 2.7 pokazuje, jak ta niepewność przenosi się na wynik.

**VaR wzorem przybliżonym** z 2.7 daje $10\,984$ PLN wobec $10\,743$ PLN z Monte Carlo, czyli różnica wynosi ok. $2\%$. Wzór zakłada normalny zwrot portfela, a w symulacji ceny są log-normalne. Rozkład wartości portfela jest więc lekko skośny w prawo, jego lewy ogon jest nieco krótszy, a VaR mniejszy. Przewaga Monte Carlo ujawnia się dopiero przy nieliniowych wypłatach, grubych ogonach i wypłatach zależnych od ścieżki.

VaR jest kwantylem, więc jego błąd losowy szacuję bootstrapem (2.6): $500$ razy losuję ze zwracaniem $50\,000$ wyników symulacji i za każdym razem liczę VaR. Podaję też **Expected Shortfall** (średnią stratę w najgorszych $5\%$ scenariuszy), który lepiej opisuje ogon rozkładu niż sam kwantyl.

In [ ]:
rng_boot = np.random.default_rng(SEED_REAL)
var_boot = np.array(
    [value_at_risk(rng_boot.choice(final_real, size=final_real.size), CAPITAL, ALPHA) for _ in range(500)]
)
lo, hi = np.percentile(var_boot, [2.5, 97.5])

tail = final_real[final_real <= np.quantile(final_real, ALPHA)]
es = CAPITAL - tail.mean()

print(f"VaR {1 - ALPHA:.0%}: {var95:,.0f} PLN, 95% przedział bootstrapowy: [{lo:,.0f}; {hi:,.0f}] PLN")
print(f"Expected Shortfall {1 - ALPHA:.0%}: {es:,.0f} PLN")


### 3.5 Backtest: czy prognozowany VaR się sprawdza?

Sama symulacja nie mówi, czy model jest dobry. Sprawdzam to na danych, których nie użyłem do kalibracji. Dla kolejnych, niezachodzących na siebie okien robię to samo:

1. kalibruję model na ostatnich $500$ sesjach,
2. symuluję $21$ sesji naprzód i wyznaczam $\mathrm{VaR}_{95\%}$,
3. porównuję z **rzeczywistym** zwrotem portfela w tych $21$ sesjach.

Jeśli model jest poprawny, strata przekracza prognozowany VaR średnio w $5\%$ okien. Liczba przekroczeń w $m$ oknach ma wtedy rozkład dwumianowy $\mathrm{Bin}(m,0{,}05)$, więc mogę sprawdzić, czy obserwowana liczba istotnie odbiega od oczekiwanej. Jest to uproszczona wersja klasycznego testu proporcji przekroczeń.

In [ ]:
bt = rolling_var_backtest(prices, weights, window=BT_WINDOW, horizon=HORIZON, alpha=ALPHA)

n_obs, n_breach = len(bt), int(bt["breach"].sum())
test = st.binomtest(n_breach, n_obs, p=ALPHA)
print(f"Okna prognoz: {n_obs}, przekroczenia VaR: {n_breach} ({n_breach / n_obs:.1%}), oczekiwane: {ALPHA:.0%}")
print(f"Test dwumianowy: p-value = {test.pvalue:.3f}")

breaches = bt[bt["breach"]]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(bt.index, -bt["var"], label="próg straty (−VaR 95%)")
ax.plot(bt.index, bt["realised_return"], "o", ms=4, label="zrealizowany zwrot")
ax.scatter(breaches.index, breaches["realised_return"], color="red", zorder=3, label="przekroczenie")
ax.axhline(0, color="black", lw=0.5)
ax.set(title="Backtest VaR: 21-sesyjny horyzont", ylabel="zwrot portfela")
ax.legend()
plt.show()

### 3.6 Wnioski i ograniczenia

Odsetek przekroczeń VaR wyniósł $4/70\approx5{,}7\%$, czyli tyle, ile oczekiwałem ($5\%$), a test dwumianowy nie odrzuca modelu ($p=0{,}78$). Cztery uwagi ograniczają jednak ten wniosek.

Po pierwsze, trzy z czterech przekroczeń przypadają na 2022 r. (inflacja, podwyżki stóp i wojna w Ukrainie), więc przekroczenia są **skupione w czasie**, a kryzysy przychodzą falami. To narusza założenie niezależności, na którym opiera się test dwumianowy, i wymagałoby osobnego testu niezależności przekroczeń.

Po drugie, pierwsze prognozy są liczone na oknie zawierającym wiosnę 2020 r. (krach związany z pandemią), stąd próg VaR rzędu $-13\%$ do $-14\%$, który w 2026 r. spada do ok. $-7\%$, gdy krach opuszcza okno kalibracji. Prognoza ryzyka zależy więc silnie od długości okna.

Po trzecie, przy $70$ oknach test wykrywa prawdziwy odsetek przekroczeń $10\%$ tylko w ok. $40\%$ przypadków, więc **brak odrzucenia nie dowodzi poprawności modelu**.

Po czwarte, brak nadwyżki przekroczeń nie przeczy wnioskom z 3.3. Zwrot miesięczny jest sumą ok. $21$ zwrotów dziennych, więc zbliża się do rozkładu normalnego (CLT), a poziom $95\%$ leży dość blisko środka rozkładu. Grube ogony ujawniłyby się wyraźniej na poziomie $99\%$, ale przy $70$ oknach oczekiwałbym wtedy mniej niż jednego przekroczenia, więc test nie miałby żadnej mocy.

Główne ograniczenia przyjętego modelu:

- zwroty są normalne i niezależne w czasie, tymczasem zmienność na rynku zmienia się w czasie i zwroty mają grube ogony,
- parametry są stałe w obrębie okna kalibracji,
- pomijam prowizje i poślizg cenowy,
- wybrałem cztery spółki, które są dziś dużymi, notowanymi emitentami, więc próbka może być zbyt korzystna (efekt przeżywalności),
- estymacja $\mu$ jest bardzo niepewna (3.4).

Naturalne rozszerzenia to rozkład t-Studenta zamiast normalnego, losowanie z historycznych zwrotów (bootstrap) albo model ze zmienną zmiennością. Z punktu widzenia projektu ważniejsze jest, że mam teraz **działający symulator rynku skalibrowany na prawdziwych danych**. W kolejnym etapie posłuży on jako środowisko, w którym agent RL będzie wybierał wagi portfela.

## Bibliografia

- Aroussi, R. *yfinance*. https://github.com/ranaroussi/yfinance.
- Billingsley, P. (1995). *Probability and Measure*, 3rd ed. New York: Wiley.
- Black, F., Scholes, M. (1973). The pricing of options and corporate liabilities. *Journal of Political Economy*, 81(3), 637–654.
- Boyle, P. P. (1977). Options: A Monte Carlo approach. *Journal of Financial Economics*, 4(3), 323–338.
- Christoffersen, P. F. (1998). Evaluating interval forecasts. *International Economic Review*, 39(4), 841–862.
- Cont, R. (2001). Empirical properties of asset returns: stylized facts and statistical issues. *Quantitative Finance*, 1(2), 223–236.
- Glasserman, P. (2004). *Monte Carlo Methods in Financial Engineering*. New York: Springer.
- Hilpisch Y. J., Reinforcement Learning for Finance: A Python-Based Introduction, O'Reilly Media, 2024.
- Golub, G. H., Van Loan, C. F. (2013). *Matrix Computations*, 4th ed. Baltimore: Johns Hopkins University Press.
- Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*, 9(3), 90–95.
- Jorion, P. (2007). *Value at Risk: The New Benchmark for Managing Financial Risk*, 3rd ed. New York: McGraw-Hill.
- Kupiec, P. H. (1995). Techniques for verifying the accuracy of risk measurement models. *The Journal of Derivatives*, 3(2), 73–84.
- Metropolis, N., Ulam, S. (1949). The Monte Carlo method. *Journal of the American Statistical Association*, 44(247), 335–341.
- Yahoo Finance. *Historical prices*. https://finance.yahoo.com.